# Evaluation usage examples

This page contains examples for using the evaluation module.

[See full module documentation here](project:/apidocs/pogg/pogg.evaluation.evaluation.md)

The evaluation module contains the following classes, each useful for storing evaluation information for the relevant type of object:

- `POGGNodeEvaluation` -- evaluation information about an individual node
- `POGGEdgeEvaluation` -- evaluation information about an individual edge
- `POGGGraphEvaluation` -- evaluation information about a graph
- `POGGEvaluation` -- evaluation information about a dataset (i.e. a set of graphs)

For the most part, interacting directly with evaluation objects is not necessary, and they are handled internally by a `POGG` object running the `run_POGG_data_to_text_algorithm` function. But this page will explain what happens under the hood for each of these objects when running the data-to-text algorithm.

Assume we have a dataset with just one graph in it:

<img src="../../../data/images/cake_graph.png" alt="Directed graph with a root node labeled 'cake' with an edge labeled 'flavor' pointing to a child node labeled 'vanilla' and another edge labeled 'filling' pointing to a child node labeled 'raspberry'. The node labeled 'raspberry' has an edge labeled 'state' pointing to a child node labeled 'fresh'." width="40%" height="auto">

A potential lexicalization of this graph could be *The vanilla cake with fresh raspberry filling*.

Also assume we have the following lexicon:

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        },
        "raspberry": {
            "comp_fxn": "noun",
            "predicate": "_raspberry_n_1",
            "intrinsic_variable_properties": {}
        },
        "vanilla": {
            "comp_fxn": "noun",
            "predicate": "_vanilla_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        },
        "state": {
            "comp_fxn": "prenominal_adjective",
            "adjective_sement": "child",
            "nominal_sement": "parent"
        }
    }
}
```

Notice we have node entries in the lexicon for `cake`, `vanilla`, and `raspberry`, but not `fresh`. We also have edge entries for `flavor` and `state` but not `filling`. The result of this is that we won't be able to create a SEMENT for each part of the graph as we traverse it and our evaluation objects will help pinpoint where the failures occurred.

## Running Example
For the following examples, assume we have run the following code:

In [70]:
from pogg.pogg_routine import POGG
from pogg.my_delphin import sementcodecs
pogg_obj = POGG("../config.yml", "../just_a_cake_dataset.yml")

# run data-to-text algorithm
pogg_obj.run_POGG_data_to_text_algorithm()

NOTE: 179 passive, 402 active edges in final generation chart; built 315 passives total. [12 results]
NOTE: generated 1 / 1 sentences, avg 5703k, time 0.62765s
NOTE: transfer did 230 successful unifies and 253 failed ones


Before continuing, let's look at the final SEMENT we generated in string format as well as the generated English text results.

In [71]:
# get the graph evaluation object
graph_eval = pogg_obj.evaluation.get_graph_evaluation("vanilla_cake")

# print the SEMENT string
print(graph_eval.generated_SEMENT_string)
# print the english results
for result in graph_eval.generated_results:
    print(result)

[ TOP: h2
  INDEX: x1
  RELS: < [ compound LBL: h13 ARG0: e10 ARG1: u11 ARG2: u12 ]
          [ udef_q LBL: h8 ARG0: x5 RSTR: h6 BODY: h7 ]
          [ _vanilla_n_1 LBL: h4 ARG0: x3 ]
          [ _cake_n_1 LBL: h2 ARG0: x1 ] >
  HCONS: < h6 qeq h4 >
  EQS: < x5 eq x3 h13 eq h9 u12 eq x5 h13 eq h2 u11 eq x1 > ]
Vanilla cakes
The vanilla cakes
Vanilla cake
The vanilla cakes.
The vanilla cake
Vanilla cakes.
A vanilla cake
The vanilla cake.
Vanilla cake.
A vanilla cake.
Vanilla cake.
Vanilla cake


Notice how the SEMENT and the results don't include any information about the fresh raspberry filling. We will now go through various evaluation objects to see both why that is and how the evaluation objects stores that information for us.

## POGGNodeEvaluation

First let's compare the information in the `POGGNodeEvaluation` objects for `cake`, `raspberry`, and `fresh`.

The details of all the attributes of a `POGGNodeEvaluation` object are in the collapsible dropdown below, but we will be focusing on the following key attributes:

- `node_covered`
- `node_included`
- `generated_SEMENT` / `generated_SEMENT_string`
- `generation_COMMENT`

:::{info} Class details
:collapsible:
````{py:class} POGGNodeEvaluation
:canonical: pogg.evaluation.evaluation.POGGNodeEvaluation

```{autodoc2-docstring}  pogg.evaluation.evaluation.POGGNodeEvaluation.__init__
```

````
:::

First, let's look at the values for these attributes for the `cake` node:

In [72]:
# get 'cake' node evaluation object
cake_node_eval = graph_eval.get_node_evaluation("cake")

# print relevant attributes
print(f"node_covered: {cake_node_eval.node_covered}")
print(f"node_included: {cake_node_eval.node_included}")
print(f"generated_SEMENT_string:\n{cake_node_eval.generated_SEMENT_string}")
print(f"generation_comment: {cake_node_eval.generation_comment}")

node_covered: True
node_included: True
generated_SEMENT_string:
[ TOP: h2
  INDEX: x1
  RELS: < [ _cake_n_1 LBL: h2 ARG0: x1 ] > ]
generation_comment: None


Here we see the following:

1. The node was covered (i.e. a SEMENT was generated from the node)
2. The node's semantic contributions were included in the final result (which we saw with the final SEMENT above that included the predication for *cake*)
3. The actual SEMENT that was generated (encoded as a string)
4. No generation comment because no failure occurred

We can contrast this with the information from the `raspberry` and `fresh` nodes:

In [73]:
# get 'raspberry' node evaluation object
raspberry_node_eval = graph_eval.get_node_evaluation("raspberry")

# print relevant attributes
print(f"node_covered: {raspberry_node_eval.node_covered}")
print(f"node_included: {raspberry_node_eval.node_included}")
print(f"generated_SEMENT_string:\n{raspberry_node_eval.generated_SEMENT_string}")
print(f"generation_comment: {raspberry_node_eval.generation_comment}")

node_covered: True
node_included: False
generated_SEMENT_string:
[ TOP: h15
  INDEX: x14
  RELS: < [ _raspberry_n_1 LBL: h15 ARG0: x14 ] > ]
generation_comment: Successor of failed element


The `raspberry` node *does* have a SEMENT associated with it, but as we can see from the `node_included` attribute, its semantic contribution was not included in the final result. The `generation_comment` (`"Successor of failed element"`) explains why. It doesn't give specifics about the failed element, but we will see which element that is later.

In [74]:
# get 'fresh' node evaluation object
fresh_node_eval = graph_eval.get_node_evaluation("fresh")

# print relevant attributes
print(f"node_covered: {fresh_node_eval.node_covered}")
print(f"node_included: {fresh_node_eval.node_included}")
print(f"generated_SEMENT_string:\n{fresh_node_eval.generated_SEMENT_string}")
print(f"generation_comment: {fresh_node_eval.generation_comment}")

node_covered: False
node_included: False
generated_SEMENT_string:
None
generation_comment: 'fresh' not in lexicon's node entries; Successor of failed element


Here we see that `node_covered` is `False`, meaning no SEMENT was generated for the node. This is reflected both in the fact that `generated_SEMENT_string` is `None` and that we have a `generation_comment` explaining the points of failure. Specifically, the lexicon key specified for the `fresh` node (`'fresh'`) isn't in the lexicon. Additionally, it's the successor of a faile element (since it's the successor to `raspberry` which itself is the successor of failed element).

The information about each node in a graph will be included in the `[graph_name]_evaluation.txt` file in the `evaluation` directory specified in the `dataset.yml` file.

Now let's look at the evaluation information for the edges in the graph.

## POGGEdgeEvaluation

Here we will look at evaluation information for every edge in the graph (i.e. `flavor`, `filling`, and `state`).

The details of all the attributes of a `POGGEdgeEvaluation` object are in the collapsible dropdown below, but we will be focusing on the following key attributes:

- `edge_covered`
- `edge_included`
- `generated_SEMENT` / `generated_SEMENT_string`
- `generation_COMMENT`

:::{info} Class details
:collapsible:
````{py:class} POGGEdgeEvaluation
:canonical: pogg.evaluation.evaluation.POGGEdgeEvaluation

```{autodoc2-docstring}  pogg.evaluation.evaluation.POGGEdgeEvaluation.__init__
```

````
:::

First, let's look at the values for these attributes for the `flavor` edge:

In [75]:
# get 'flavor' edge evaluation object
# must pass in parent name, child name, and edge data to single out the edge
flavor_edge_eval = graph_eval.get_edge_evaluation("cake", "vanilla", {"label" : "flavor"})

# print relevant attributes
print(f"edge_covered: {flavor_edge_eval.edge_covered}")
print(f"edge_included: {flavor_edge_eval.edge_included}")
print(f"generated_SEMENT_string:\n{flavor_edge_eval.generated_SEMENT_string}")
print(f"generation_comment: {flavor_edge_eval.generation_comment}")

edge_covered: True
edge_included: True
generated_SEMENT_string:
[ TOP: h2
  INDEX: x1
  RELS: < [ compound LBL: h13 ARG0: e10 ARG1: u11 ARG2: u12 ]
          [ udef_q LBL: h8 ARG0: x5 RSTR: h6 BODY: h7 ]
          [ _vanilla_n_1 LBL: h4 ARG0: x3 ]
          [ _cake_n_1 LBL: h2 ARG0: x1 ] >
  HCONS: < h6 qeq h4 >
  EQS: < x5 eq x3 h13 eq h9 u12 eq x5 h13 eq h2 u11 eq x1 > ]
generation_comment: None


`edge_covered` is `True`, telling us that the SEMENTs for the parent and child were successfully used in the composition of a new SEMENT that includes both of their semantic contributions. `edge_included` is also `True`, telling us that the edge's semantic contributions are included in the final result. We can also see the SEMENT that was created by composing the SEMENTs from the parent and child nodes in a compound noun construction.

:::{info} Semantic contributions of nodes vs. edges
:collapsible:

Note that when it comes to nodes it's fairly easy to detect at a glance whether a particular node made semantic contributions to the result. For example, we can clearly see that `cake` contributed because of the `_cake_n_1` predication in the RELS list. For edges, however, it's more difficult to see because the semantic contribution of edges is to ensure two SEMENTs are properly composed. Here the edge adds some necessary predications for the compound noun construction (`compound` and `udef_q`) but this won't always be the case. For example, composition between a noun and an adjective just involves plugging the adjective's `ARG1` slot with the intrinsic variable of the noun, so there won't be "extra" predicates contributed by the edge.

:::

Now let's look at the `filling` edge.

In [76]:
# get 'filling' edge evaluation object
# must pass in parent name, child name, and edge data to single out the edge
filling_edge_eval = graph_eval.get_edge_evaluation("cake", "raspberry", {"label" : "filling"})

# print relevant attributes
print(f"edge_covered: {filling_edge_eval.edge_covered}")
print(f"edge_included: {filling_edge_eval.edge_included}")
print(f"generated_SEMENT_string:\n{filling_edge_eval.generated_SEMENT_string}")
print(f"generation_comment: {filling_edge_eval.generation_comment}")

edge_covered: False
edge_included: False
generated_SEMENT_string:
None
generation_comment: 'filling' not in lexicon's edge entries


This edge is not covered or included so no SEMENT is associated with it. The `generation_comment` tells us that the reason for the failure is that there is no entry in the lexicon for `filling` so the algorithm was not able to determine what type of composition this edge is supposed to contribute. This failure is also what causes the `raspberry` node to fail, hence the `"Successor of failed element"` comment in that node's evaluation object.

Lastly, let's look at the `state` edge.

In [77]:
# get 'state' edge evaluation object
# must pass in parent name, child name, and edge data to single out the edge
state_edge_eval = graph_eval.get_edge_evaluation("raspberry", "fresh", {"label" : "state"})

# print relevant attributes
print(f"edge_covered: {state_edge_eval.edge_covered}")
print(f"edge_included: {state_edge_eval.edge_included}")
print(f"generated_SEMENT_string:\n{state_edge_eval.generated_SEMENT_string}")
print(f"generation_comment: {state_edge_eval.generation_comment}")

edge_covered: False
edge_included: False
generated_SEMENT_string:
None
generation_comment: child of 'state' has no SEMENT


This edge also is not covered or included, except here the reason is not that `state` isn't in the lexicon, since it is, but rather that the child node (`fresh`) produced no SEMENT making composition impossible. We saw above that `fresh` didn't produce a SEMENT because `fresh` wasn't in the lexicon, and here we see how that impacts other elements in the graph.

## POGGGraphEvaluation

Now that we've looked at some of the evaluation objects for individual graph elements, we can look at some of the aggregate information included in a `POGGGraphEvaluation` object.

The details of all the attributes of a `POGGGraphEvaluation` object are in the collapsible dropdown below, but we will be focusing on the following key attributes:

- `node_coverage`
- `node_inclusion`
- `edge_coverage`
- `edge_inclusion`
- `generated_SEMENT` / `generated_SEMENT_string`
- `collapsed_SEMENT` / `collapsed_SEMENT_string`
- `prepped_SEMENT` / `prepped_SEMENT_string`
- `generated_results`

:::{info} Class details
:collapsible:
````{py:class} POGGGraphEvaluation
:canonical: pogg.evaluation.evaluation.POGGGraphEvaluation

```{autodoc2-docstring}  pogg.evaluation.evaluation.POGGGraphEvaluation.__init__
```

````
:::

First, let's look at the coverage and inclusion values for both nodes and edges for this graph.

In [78]:
# print coverage and inclusion values
print(f"node_coverage: {graph_eval.node_coverage}")
print(f"node_inclusion: {graph_eval.node_inclusion}")
print(f"edge_coverage: {graph_eval.edge_coverage}")
print(f"edge_inclusion: {graph_eval.edge_inclusion}")

node_coverage: 0.75
node_inclusion: 0.5
edge_coverage: 0.3333333333333333
edge_inclusion: 0.3333333333333333


These are the numbers we expected since we know that 3 out of the 4 nodes were covered, but only 2 of them contributed to the final result. As far as edges, only 1 out of 3 edges was covered and also included in the result.

Now let's look at the different SEMENT strings and compare the differences.

In [79]:
# print SEMENTs encoded as strings
print(f"generated_SEMENT_string:\n{graph_eval.generated_SEMENT_string}\n")
print(f"collapsed_SEMENT_string:\n{graph_eval.collapsed_SEMENT_string}\n")
print(f"prepped_SEMENT_string:\n{graph_eval.prepped_SEMENT_string}\n")

generated_SEMENT_string:
[ TOP: h2
  INDEX: x1
  RELS: < [ compound LBL: h13 ARG0: e10 ARG1: u11 ARG2: u12 ]
          [ udef_q LBL: h8 ARG0: x5 RSTR: h6 BODY: h7 ]
          [ _vanilla_n_1 LBL: h4 ARG0: x3 ]
          [ _cake_n_1 LBL: h2 ARG0: x1 ] >
  HCONS: < h6 qeq h4 >
  EQS: < x5 eq x3 h13 eq h9 u12 eq x5 h13 eq h2 u11 eq x1 > ]

collapsed_SEMENT_string:
[ TOP: h13
  INDEX: x1
  RELS: < [ compound LBL: h13 ARG0: e10 ARG1: x1 ARG2: x3 ]
          [ udef_q LBL: h8 ARG0: x3 RSTR: h6 BODY: h7 ]
          [ _vanilla_n_1 LBL: h4 ARG0: x3 ]
          [ _cake_n_1 LBL: h13 ARG0: x1 ] >
  HCONS: < h6 qeq h4 > ]

prepped_SEMENT_string:
[ TOP: h24
  INDEX: e21
  RELS: < [ unknown LBL: h20 ARG: x1 ARG0: e21 ]
          [ def_udef_a_q LBL: h19 ARG0: x1 RSTR: h17 BODY: h18 ]
          [ compound LBL: h13 ARG0: e10 ARG1: x1 ARG2: x3 ]
          [ udef_q LBL: h8 ARG0: x3 RSTR: h6 BODY: h7 ]
          [ _vanilla_n_1 LBL: h4 ARG0: x3 ]
          [ _cake_n_1 LBL: h13 ARG0: x1 ] >
  HCONS: < h6 qeq h

The graph evaluation object stores three different versions of the generated SEMENT along with their string encodings:

- `generated_SEMENT` -- the original SEMENT generated for the graph
- `collapsed_SEMENT` -- the SEMENT with the equalities collapsed to one representative value
    - Notice how in the original SEMENT there's an entry in the EQs list stating that `x5 = x3` (that is, the ARG0 of `udef_q` is identified with the `ARG0` of `vanilla`), but in the collapsed version both of these slots simply have the same value of `x3`, making it easier to read.
- `prepped_SEMENT` -- a modified version of the SEMENT prepped for generation by the ERG
    - There are certain constraints that a semantic structure must meet for the ERG to be able to generate from it[^1] so the data-to-text algorithm checks that these are met and creates a new version of the SEMENT that meets them if necessary.


[^1]: See the documentation for the [`prepare_for_generation` function](project:/apidocs/pogg/pogg.semantic_composition.semantic_algebra.md#pogg.semantic_composition.semantic_algebra.SemanticAlgebra.prepare_for_generation)) in the `SemanticAlgebra` class for the specific constraints.

Lastly let's look at the text results that were generated from the `prepped_SEMENT`:

In [80]:
# print English text results
for result in graph_eval.generated_results:
    print(result)

Vanilla cakes
The vanilla cakes
Vanilla cake
The vanilla cakes.
The vanilla cake
Vanilla cakes.
A vanilla cake
The vanilla cake.
Vanilla cake.
A vanilla cake.
Vanilla cake.
Vanilla cake


## POGGEvaluation

The last evaluation object type to cover is the `POGGEvaluation` object which stores evaluation information for a whole dataset.

The details of all the attributes of a `POGGEvaluation` object are in the collapsible dropdown below, but we will be focusing on the following key attributes:

- `graph_SEMENT_coverage`
- `graph_text_coverage`
- `full_node_coverage`
- `full_node_inclusion`
- `full_edge_coverage`
- `full_edge_inclusion`

:::{info} Class details
:collapsible:
````{py:class} POGGEvaluation
:canonical: pogg.evaluation.evaluation.POGGEvaluation

```{autodoc2-docstring}  pogg.evaluation.evaluation.POGGEvaluation.__init__
```

````
:::

In order to make these metrics slightly more interesting, let's add another graph to our dataset:

<img src="../../../data/images/cupcake_graph.png" alt="Directed graph with a root node labeled 'cupcake' with an edge labeled 'flavor' pointing to a child node labeled 'vanilla'." width="40%" height="auto">

A possible lexicalization for this simple graph could be *The vanilla cupcake*. As a reminder, assume we are still using the same lexicon as in the previous examples:

:::{code} Lexicon
:collapsible:

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        },
        "raspberry": {
            "comp_fxn": "noun",
            "predicate": "_raspberry_n_1",
            "intrinsic_variable_properties": {}
        },
        "vanilla": {
            "comp_fxn": "noun",
            "predicate": "_vanilla_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        },
        "state": {
            "comp_fxn": "prenominal_adjective",
            "adjective_sement": "child",
            "nominal_sement": "parent"
        }
    }
}
```
:::

Note that `cupcake` is not one of the lexicon keys, so this graph will not generate a SEMENT and as a result won't generate any text results.

Let's rerun the data-to-text algorithm now that we have added another graph to the dataset:

In [81]:
# this dataset now has two graphs in it
pogg_obj = POGG("../config.yml", "../just_a_cake_dataset.yml")

# run data-to-text algorithm
pogg_obj.run_POGG_data_to_text_algorithm()

NOTE: 179 passive, 402 active edges in final generation chart; built 315 passives total. [12 results]
NOTE: generated 1 / 1 sentences, avg 5703k, time 0.49182s
NOTE: transfer did 230 successful unifies and 253 failed ones


And let's take a peek at some of the `POGGGraphEvaluation` metrics before looking at the metrics for the whole dataset:

In [82]:
# get the graph evaluation object
graph_eval = pogg_obj.evaluation.get_graph_evaluation("vanilla_cupcake")

# print coverage and inclusion values
print(f"node_coverage: {graph_eval.node_coverage}")
print(f"node_inclusion: {graph_eval.node_inclusion}")
print(f"edge_coverage: {graph_eval.edge_coverage}")
print(f"edge_inclusion: {graph_eval.edge_inclusion}\n")

# print SEMENTs encoded as strings
print(f"generated_SEMENT_string:\n{graph_eval.generated_SEMENT_string}\n")
print(f"collapsed_SEMENT_string:\n{graph_eval.collapsed_SEMENT_string}\n")
print(f"prepped_SEMENT_string:\n{graph_eval.prepped_SEMENT_string}\n")

# print English text results
for result in graph_eval.generated_results:
    print(result)

node_coverage: 0.5
node_inclusion: 0.0
edge_coverage: 0.0
edge_inclusion: 0.0

generated_SEMENT_string:
None

collapsed_SEMENT_string:
None

prepped_SEMENT_string:
None



As expected, we don't have any SEMENT results and therefore no English text results either. As far as our quantitative metrics, the only one that isn't 0 is `node_coverage` because a SEMENT was generated for the `vanilla` child node, but then semantic composition couldn't be performed between the parent and child using the edge since the parent had no SEMENT associated with it.

Now let's look at the metrics of interest in the `POGGEvaluation` object for the whole dataset which is stored in the `evaluation` attribute for the `POGG` object:

In [83]:
# print coverage and inclusion values
print(f"graph_SEMENT_coverage (what % of graphs generated a SEMENT?): {pogg_obj.evaluation.graph_SEMENT_coverage}")
print(f"graph_text_coverage (what % of graphs generated English text results?): {pogg_obj.evaluation.graph_SEMENT_coverage}")
print(f"full_node_coverage: {pogg_obj.evaluation.full_node_coverage}")
print(f"full_node_inclusion: {pogg_obj.evaluation.full_node_inclusion}")
print(f"full_edge_coverage: {pogg_obj.evaluation.full_edge_coverage}")
print(f"full_edge_inclusion: {pogg_obj.evaluation.full_edge_inclusion}")

graph_SEMENT_coverage (what % of graphs generated a SEMENT?): 0.5
graph_text_coverage (what % of graphs generated English text results?): 0.5
full_node_coverage: 0.6666666666666666
full_node_inclusion: 0.3333333333333333
full_edge_coverage: 0.25
full_edge_inclusion: 0.25


Here we see that 50% of the graphs generated a SEMENT (1 out of the 2, as expected) and 50% of the graphs generated English text results (1 out of the 2, as expected). We also see the details of what percentage nodes and edges across the entire dataset were covered/included.

## Conclusion

As stated at the top of this guide, a user typically will not be dealing with the individual evaluation objects and the data-to-text algorithm will take care of marking nodes/edges as covered/included where appropriate and calculating the final metrics at the end of the run. But once the algorithm is finished running on a dataset the information in these objects can help pinpoint where failures occurred in the conversion process.

In order to better facilitate combing through the data, the data-to-text algorithm also calls functions from the [`reporting` module](project:/apidocs/pogg/pogg.evaluation.reporting.md) to print summaries for each graph in the dataset as well as the dataset as a whole. See the [usage guide on the `reporting` module](project:/usage_nbs/pogg/evaluation/reporting_usage.ipynb) for examples of those reports.